In [11]:
# STEP 0 — Setup

import numpy as np
import pandas as pd
from pathlib import Path

# Folder where your Excel files are and where you will save your outputs
DATA_DIR = Path("Data_2026")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

# Check that the data folder exists
assert DATA_DIR.exists(), "❌ I cannot find the folder Data_2026. Check folder name/location in VS Code."
print("✅ Setup OK")

✅ Setup OK


In [12]:
# STEP 1 — Static file -> Emerging Markets universe (ISIN list)

static = pd.read_excel(DATA_DIR / "Static_2025.xlsx")

# Clean text columns (remove spaces)
static["ISIN"] = static["ISIN"].astype(str).str.strip()
static["Region"] = static["Region"].astype(str).str.strip()

# Emerging Markets ( my group )
em_isins = static.loc[static["Region"] == "EM", "ISIN"].dropna().unique().tolist()

print("✅ EM ISINs:", len(em_isins))

✅ EM ISINs: 702


In [13]:
# STEP 2 — Monthly Return Index --> Monthly returns (no forward-fill)

def load_monthly_ri(filepath):
    """
    Reads a Datastream monthly RI Excel file shaped like:
    ISIN | NAME | date1 | date2 | ...
    Returns a DataFrame:
    rows = dates
    columns = ISINs
    """
    raw = pd.read_excel(filepath)

    # Remove rows with missing ISIN (Datastream error rows)
    raw = raw.dropna(subset=["ISIN"])

    # Clean ISIN
    raw["ISIN"] = raw["ISIN"].astype(str).str.strip()

    # Keep only date columns (everything except ISIN + NAME)
    cols_to_drop = [c for c in ["ISIN", "NAME"] if c in raw.columns]
    date_cols = [c for c in raw.columns if c not in cols_to_drop]

    # Build RI table: rows=ISIN, cols=dates
    df = raw.set_index("ISIN")[date_cols]
    df = df.apply(pd.to_numeric, errors="coerce")

    # Transpose: rows=dates, cols=ISIN
    df = df.T
    df.index = pd.to_datetime(df.index, errors="coerce")
    df = df.sort_index()

    return df

# Load all firms RI then keep EM firms
ri_all = load_monthly_ri(DATA_DIR / "DS_RI_T_USD_M_2025.xlsx")
ri_em = ri_all.loc[:, ri_all.columns.intersection(em_isins)]

# Low RI rule: RI < 0.5 -> missing
ri_em = ri_em.mask(ri_em < 0.5)

# Monthly returns WITHOUT hidden filling
ret_em = ri_em.pct_change(fill_method=None)

print("✅ Returns built:", ret_em.shape)

✅ Returns built: (314, 702)


In [14]:
# STEP 3 — Delisting: set next-month return = -100% if stock disappears forever

last_date = ri_em.index.max()
delist_count = 0

for isin in ri_em.columns:
    s = ri_em[isin]

    lv = s.last_valid_index()
    if pd.isna(lv) or lv == last_date:
        continue

    # Check it never comes back after lv
    after = s.loc[lv:]
    if after.notna().sum() == 1:  # only the last point itself is valid
        pos = ri_em.index.get_loc(lv)
        if pos + 1 < len(ri_em.index):
            next_date = ri_em.index[pos + 1]
            ret_em.loc[next_date, isin] = -1.0
            delist_count += 1

print("✅ Delistings added:", delist_count)

✅ Delistings added: 62


In [15]:
# STEP 4 — Keep only stocks with at least 36 valid monthly returns ( 3 years )

MIN_MONTHS = 36

valid_months = ret_em.notna().sum(axis=0)
keep_isins = valid_months[valid_months >= MIN_MONTHS].index

ret_final = ret_em.loc[:, keep_isins].copy()

print("✅ Stocks before:", ret_em.shape[1])
print("✅ Stocks after 36-month filter:", ret_final.shape[1])

✅ Stocks before: 702
✅ Stocks after 36-month filter: 660


In [16]:
# STEP 5 — Annual emissions (Scope 1 + Scope 2)

def load_annual_wide(filepath):
    """
    Reads files shaped like:
    NAME | ISIN | 1999 | 2000 | ... | 2025
    Returns:
    rows = years
    columns = ISINs
    """
    raw = pd.read_excel(filepath)
    raw = raw.dropna(subset=["ISIN"])
    raw["ISIN"] = raw["ISIN"].astype(str).str.strip()

    year_cols = [c for c in raw.columns if isinstance(c, (int, float))]
    df = raw.set_index("ISIN")[year_cols]
    df = df.apply(pd.to_numeric, errors="coerce")

    df = df.T
    df.index = df.index.astype(int)
    df = df.sort_index()
    return df

s1_all = load_annual_wide(DATA_DIR / "DS_CO2_SCOPE_1_Y_2025.xlsx")
s2_all = load_annual_wide(DATA_DIR / "DS_CO2_SCOPE_2_Y_2025.xlsx")

s1_em = s1_all.loc[:, s1_all.columns.intersection(em_isins)].ffill()
s2_em = s2_all.loc[:, s2_all.columns.intersection(em_isins)].ffill()

E_s12 = s1_em + s2_em

print("✅ E_s12 built:", E_s12.shape)

✅ E_s12 built: (27, 702)


In [17]:
# STEP 6 — Annual revenue + Carbon intensity

rev_all = load_annual_wide(DATA_DIR / "DS_REV_Y_2025.xlsx")
rev_em = rev_all.loc[:, rev_all.columns.intersection(em_isins)].ffill()

# Remove zero/negative revenue
rev_em = rev_em.mask(rev_em <= 0)

ci_em = E_s12 / rev_em

print("✅ CI built:", ci_em.shape)

✅ CI built: (27, 702)


In [18]:
# STEP 7 — Align CI to the final return universe (same columns, same order)

ci_final = ci_em.reindex(columns=ret_final.columns)

print("✅ Final returns universe:", ret_final.shape[1])
print("✅ CI universe after align:", ci_final.shape[1])

✅ Final returns universe: 660
✅ CI universe after align: 660


In [19]:
# STEP 8 — Compute mean returns and covariance matrix (monthly + annualized)

R = ret_final.copy()

# Mean monthly return per stock
mu_m = R.mean(axis=0, skipna=True)

# Monthly covariance matrix (pairwise deletion for NaNs)
Sigma_m = R.cov()

# Optional annualized versions (common in finance)
mu_a = (1 + mu_m)**12 - 1
Sigma_a = Sigma_m * 12

print("✅ mu_m length:", len(mu_m))
print("✅ Sigma_m shape:", Sigma_m.shape)

✅ mu_m length: 660
✅ Sigma_m shape: (660, 660)


In [20]:
# STEP 9 — Export a compact Excel file for checking

from pathlib import Path

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

N_COLS = 100
sample_cols = list(ret_final.columns[:N_COLS])

# Samples to export
returns_sample = ret_final.loc[:, sample_cols]
ri_sample      = ri_em.loc[:, sample_cols]     # monthly RI (cleaned)
emiss_sample   = E_s12.loc[:, sample_cols]     # annual emissions S1+S2
rev_sample     = rev_em.loc[:, sample_cols]    # annual revenue
ci_sample      = ci_final.loc[:, sample_cols]  # annual carbon intensity

# Small summary sheet (useful)
summary = pd.DataFrame({
    "metric": [
        "EM ISINs (Static)",
        "Stocks after 36-month filter",
        "Months in returns sample",
        "Delisting events inserted",
        "Missing % in returns (final)",
        "Missing % in CI (final)",
    ],
    "value": [
        len(em_isins),
        ret_final.shape[1],
        ret_final.shape[0],
        delist_count,
        float(ret_final.isna().mean().mean()),
        float(ci_final.isna().mean().mean()),
    ],
})

output_path = OUT_DIR / "FINAL_CHECK_ROBERTO.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="summary", index=False)
    returns_sample.to_excel(writer, sheet_name="returns_monthly")
    ri_sample.to_excel(writer, sheet_name="ri_monthly")
    emiss_sample.to_excel(writer, sheet_name="emissions_S1S2")
    rev_sample.to_excel(writer, sheet_name="revenue")
    ci_sample.to_excel(writer, sheet_name="carbon_intensity")

print("✅ Excel file created:", output_path)

✅ Excel file created: outputs/FINAL_CHECK_ROBERTO.xlsx
